# RD01-b — Residual regime discovery and online state confirmation

**Goal:** determine whether interpretable, duration-aware tree states explain next-5/15/30/60m *signed returns* beyond direct feature regression and explicit interactions. **Primary horizon: 30 minutes.**

Uses CS04-A causal data, RD01 chronology, and RD01's trailing variance proxy—not AM-T-HAR-Blend's forecast. Three-consecutive-bar state confirmation induces dwell mechanically and must **not** be treated as evidence of structural persistence. The repeatedly inspected historical periods are for discovery only.

Frozen protocol: `discussions/Regime_Discovery/001b_chatgpt_residual_regime_preregistration.md`.

In [1]:
import os,sys,json,time,tempfile,subprocess,gc
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Open JupyterLab in your repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import observable_regime_discovery as rd
from data_engineering.modeling import observable_regime_discovery_b as rb
SYNTHETIC=os.getenv('RD01B_SYNTHETIC','0')=='1'
cfg=rb.Spec(minimum_bootstrap_days=5,bootstrap_repetitions=25,tree_min_leaf_absolute=25) if SYNTHETIC else rb.Spec()
if SYNTHETIC:
    temp=tempfile.TemporaryDirectory(prefix='rd01b-')
    OUT=Path(temp.name)/'01b'
else:
    rootval=os.getenv('DERIBIT_DATA_ROOT')
    if not rootval:raise RuntimeError('DERIBIT_DATA_ROOT not set')
    STORE=Path(rootval)/'research'/'cs04a_feature_store'
    OUT=ROOT/'notebooks'/'Regime_Discovery'/'01b'
OUT.mkdir(parents=True,exist_ok=True)
print('RD01-b',rb.VERSION,'synthetic',SYNTHETIC,'output',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
r=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_observable_regime_discovery_b.py')],cwd=str(ROOT),env=env,text=True,capture_output=True)
print(r.stdout[-1300:]);print(r.stderr[-700:]);assert r.returncode==0, 'RD01-b tests failed; stop historical scan' 

RD01-b RD01b-0.1.0 synthetic False output D:\Projects\deribit_market_data\notebooks\Regime_Discovery\01b
..........                                                               [100%]
10 passed in 40.40s




In [2]:
datasets={};excluded={}
if SYNTHETIC:
    sys.path.insert(0,str(ROOT/'tests'))
    from test_observable_regime_discovery import synthetic_panel
    datasets={'BTC':synthetic_panel(days=94)}
else:
    columns=list(dict.fromkeys(['state_time','return_5m','feature_ready','log_rv24h',*rd.BASE_COLS,*rd.TREE_COLS,'ret_5m','ret_15m','ret_30m']))
    for asset in cfg.assets:
        path=STORE/f'{asset}_features_and_forward_magnitude.parquet'
        if not path.is_file():excluded[asset]=f'Missing local feature store: {path}';continue
        try:
            datasets[asset]=pd.read_parquet(path,columns=columns)
            print(asset,'rows',len(datasets[asset]))
        except Exception as e:excluded[asset]=f'{type(e).__name__}: {e}'
assert datasets, f'No historical inputs: {excluded}'
if not SYNTHETIC and not {'BTC','ETH'}.issubset(datasets):print('WARNING: BTC/ETH primary coverage incomplete')

BTC rows 221761
ETH rows 221761
SOL rows 221761
XRP rows 221761


In [3]:
results={};coverage=[]
for asset,frame in datasets.items():
    started=time.perf_counter()
    try:
        z=rb.scan_asset(frame,cfg,asset)
        results[asset]=z
        coverage.append({'asset':asset,'complete_days':z['n_days'],'score_rows':len(z['scores']),
                         'primary_tree_structures':len(z['definitions']),
                         'elapsed_seconds':round(time.perf_counter()-started,2)})
        print(asset,'complete',len(z['scores']),'scores',len(z['definitions']),'horizon tree structures')
    except Exception as e:
        excluded[asset]=f'{type(e).__name__}: {e}'
        if not SYNTHETIC:raise
    finally:
        del frame
        gc.collect()
assert results, 'No assets scanned'
display(pd.DataFrame(coverage))

BTC complete 60 scores 12 horizon tree structures
ETH complete 60 scores 12 horizon tree structures
SOL complete 60 scores 12 horizon tree structures
XRP complete 60 scores 12 horizon tree structures


,asset,complete_days,score_rows,primary_tree_structures,elapsed_seconds
0,BTC,736,60,12,233.92
1,ETH,736,60,12,237.59
2,SOL,736,60,12,235.95
3,XRP,736,60,12,236.57


In [4]:
out=rb.consolidate(results,cfg)
out['manifest']['synthetic']=SYNTHETIC
out['manifest']['excluded']=excluded
out['manifest']['n_asset_folds']=len(out['scores'].groupby(['asset','fold']))
display(out['comparison_summary'].sort_values(['horizon_min','delta_nll_vs_direct'])[['method','horizon_min','n_asset_folds','delta_nll_vs_direct','delta_binary_vs_direct']])
print('30m per-asset-fold paired NLL against interaction Ridge:')
focus=out['bootstrap'].query("horizon_min==30 and metric=='nll' and reference=='interaction_ridge'")
display(focus[['asset','fold','method','delta','ci025','ci975','status']])
print('Persistence is mechanically influenced by the three-bar confirmation rule.')
display(out['persistence'][['asset','fold','method','observed_stay','within_day_iid_expected_stay']])
print('Descriptive decision only:',json.dumps(out['decision'],indent=2))

,method,horizon_min,n_asset_folds,delta_nll_vs_direct,delta_binary_vs_direct
8,interaction_ridge,5,12,-0.000035,0.000005
4,direct_ridge,5,12,0.000000,0.000000
12,residual_tree,5,12,0.000036,0.000027
16,residual_tree_sticky,5,12,0.000048,0.000038
0,baseline,5,12,0.000855,0.000712
5,direct_ridge,15,12,0.000000,0.000000
13,residual_tree,15,12,0.000035,0.000008
9,interaction_ridge,15,12,0.000086,0.000084
17,residual_tree_sticky,15,12,0.000300,0.000263
1,baseline,15,12,0.000396,0.000391


30m per-asset-fold paired NLL against interaction Ridge:


,asset,fold,method,delta,ci025,ci975,status
36,BTC,1,residual_tree,0.000002,-0.000184,0.000193,EXPLORATORY_UNADJUSTED
39,BTC,1,residual_tree_sticky,0.000647,-0.000156,0.001457,EXPLORATORY_UNADJUSTED
100,BTC,2,residual_tree,-0.000287,-0.000871,0.000259,EXPLORATORY_UNADJUSTED
103,BTC,2,residual_tree_sticky,0.000061,-0.000964,0.001196,EXPLORATORY_UNADJUSTED
164,BTC,3,residual_tree,-0.000749,-0.001326,-0.000228,EXPLORATORY_UNADJUSTED
167,BTC,3,residual_tree_sticky,-0.000425,-0.000864,0.000063,EXPLORATORY_UNADJUSTED
228,ETH,1,residual_tree,0.000141,-0.000320,0.000559,EXPLORATORY_UNADJUSTED
231,ETH,1,residual_tree_sticky,0.000143,-0.000931,0.001252,EXPLORATORY_UNADJUSTED
292,ETH,2,residual_tree,-0.000041,-0.000475,0.000411,EXPLORATORY_UNADJUSTED
295,ETH,2,residual_tree_sticky,0.000318,-0.000190,0.000839,EXPLORATORY_UNADJUSTED


Persistence is mechanically influenced by the three-bar confirmation rule.


,asset,fold,method,observed_stay,within_day_iid_expected_stay
0,BTC,1,residual_tree,0.505078,0.391607
1,BTC,1,residual_tree_sticky,0.955297,0.469325
2,BTC,2,residual_tree,0.522574,0.419805
3,BTC,2,residual_tree_sticky,0.954333,0.642650
4,BTC,3,residual_tree,0.715694,0.553710
5,BTC,3,residual_tree_sticky,0.941879,0.721176
6,ETH,1,residual_tree,0.356661,0.279394
7,ETH,1,residual_tree_sticky,0.974201,0.578741
8,ETH,2,residual_tree,0.441767,0.312322
9,ETH,2,residual_tree_sticky,0.966121,0.504208


Descriptive decision only: {
  "status": "HISTORICAL_DISCOVERY_NOT_CONFIRMATION",
  "primary_horizon_min": 30,
  "descriptive_candidates": [
    {
      "model": "residual_tree",
      "primary_horizon_min": 30,
      "average_delta_nll_vs_interaction": -0.0001501483607543319,
      "favorable_asset_folds": 8,
      "n_asset_folds": 12,
      "historical_candidate": false,
      "independent_confirmation": false
    },
    {
      "model": "residual_tree_sticky",
      "primary_horizon_min": 30,
      "average_delta_nll_vs_interaction": 0.00019582409545933674,
      "favorable_asset_folds": 3,
      "n_asset_folds": 12,
      "historical_candidate": false,
      "independent_confirmation": false
    }
  ],
  "no_prospective_claim": true,
  "AM_T_HAR_Blend_modified": false,
  "note": "Persistence from three-bar confirmation is mechanically induced; cannot prove structural regimes."
}


In [5]:
outputs={'01b_scores.csv':'scores','01b_paired_bootstrap.csv':'bootstrap',
    '01b_comparison_summary.csv':'comparison_summary','01b_state_forecasts.csv':'characteristics',
    '01b_persistence.csv':'persistence','01b_episode_summary.csv':'episode_summary',
    '01b_transitions.csv':'transitions','01b_tree_definitions.csv':'definitions'}
for name,key in outputs.items():out[key].to_csv(OUT/name,index=False)
pd.DataFrame(coverage).to_csv(OUT/'01b_coverage.csv',index=False)
(OUT/'01b_manifest.json').write_text(json.dumps(out['manifest'],indent=2,default=str),encoding='utf-8')
(OUT/'01b_decision.json').write_text(json.dumps(out['decision'],indent=2,default=str),encoding='utf-8')
assert all((OUT/f).is_file() for f in list(outputs)+['01b_coverage.csv','01b_manifest.json','01b_decision.json'])
print('Saved',len(outputs)+3,'aggregate outputs to',OUT)
print('Synthetic outputs are NOT market evidence. Do not commit private feature store or per-origin labels.')

Saved 11 aggregate outputs to D:\Projects\deribit_market_data\notebooks\Regime_Discovery\01b
Synthetic outputs are NOT market evidence. Do not commit private feature store or per-origin labels.


## Interpretation rules

- The main question is **incremental 30m signed-return density NLL versus interaction Ridge**, not whether sticky states persist (they are engineered to persist).
- Even a historical negative NLL delta is exploratory only. The legacy CS03b/VD18 heldout interval was previously examined.
- State labels are tree IDs, specific to **asset, fold and horizon**. No global regime names or duration claims are justified from these outputs.
- The common Student-t scale uses the RD01 trailing-RV proxy—not the AM-T-HAR-Blend conditional variance. Actual AM-T head-to-head belongs in RD03.
- The original RD01 code, previous research outputs, collectors and AM-T-HAR-Blend are unchanged.